In [1]:
# Load the Drive helper and mount:
from google.colab import drive

# This will prompt for authorization:
drive.mount('/content/drive',force_remount=False)

%cd /content/drive/MyDrive/MouseWound/Healnet_Models/DeepMapper_Analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MouseWound/Healnet_Models/DeepMapper_Analysis


In [2]:
import csv
import numpy as np
import pandas as pd
import glob

In [3]:
#!unzip deepmapper_on_testing_mouse_imgage.zip -x "__MACOSX/*" "*.DS_Store"

## Examine one .csv file

### Import Data

In [84]:
example = "./deepmapper_on_testing_mouse_imgage/adult/mouse_test_stages_seed_0.csv"
expert_labels = pd.read_csv("../Experts_Cropped_Images_Wound_Stage_Probabilities_from_table.csv")
expert_labels["label_idx"] = expert_labels[stage_cols].values.argmax(axis=1)

expert_labels["Image"] = (
    expert_labels["Image"]
    .astype(str)
    .str.replace("Young_", "", regex=False)
)

In [13]:
# load predictions .csv as list (for ease of compute later)
rows = []
with open(example, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    data = list(reader)

# print(header)
# print(data[0])

### Visualize DeepMapper Data

In [14]:
#visualize data

columns = ["Image", "hemostasis", "inflammatory", "proliferative", "maturation"]
stage_cols = columns[1:]

data_df = pd.DataFrame(data, columns=columns)

In [15]:
data_df.head()

,Image,hemostasis,inflammatory,proliferative,maturation
0,Day 0_A8-1-L.png,1.0,0.0,0.0,0.0
1,Day 1_A8-1-L.png,0.598047459842907,0.40195254015709303,0.0,0.0
2,Day 2_A8-1-L.png,0.3480915664733608,0.5435975304133234,0.10831090311331582,0.0
3,Day 3_A8-1-L.png,0.2171105787358418,0.4305718614578983,0.30074079058691094,0.05157676921934897
4,Day 4_A8-1-L.png,0.1394018418703596,0.3337643379281234,0.38189776676132814,0.14493605344018887


### Expert Labels

In [42]:
# all labels
expert_labels

,Image,hemostasis,inflammatory,proliferative,maturation,label_idx
0,Day 0_A8-5-L.png,1,0,0,0,0
1,Day 0_A8-3-L.png,1,0,0,0,0
2,Day 0_A8-1-L.png,0,1,0,0,1
3,Day 0_A8-4-L.png,1,0,0,0,0
4,Day 0_A8-4-R.png,1,0,0,0,0
...,...,...,...,...,...,...
249,Day 15_Young_Y8-3-L.png,0,0,0,1,3
250,Day 15_Young_Y8-1-R.png,0,0,0,1,3
251,Day 15_Young_Y8-3-R.png,0,0,0,1,3
252,Day 15_Young_Y8-4-R.png,0,0,0,1,3


In [43]:
# test labels
expert_labels[expert_labels['Image'].isin(data_df['Image'])]

,Image,hemostasis,inflammatory,proliferative,maturation,label_idx
2,Day 0_A8-1-L.png,0,1,0,0,1
18,Day 1_A8-1-L.png,0,1,0,0,1
34,Day 2_A8-1-L.png,0,1,0,0,1
50,Day 3_A8-1-L.png,0,1,0,0,1
66,Day 4_A8-1-L.png,0,1,0,0,1
82,Day 5_A8-1-L.png,0,1,0,0,1
98,Day 6_A8-1-L.png,0,0,1,0,2
114,Day 7_A8-1-L.png,0,0,1,0,2
130,Day 8_A8-1-L.png,0,0,1,0,2
146,Day 9_A8-1-L.png,0,0,1,0,2


### Compute Accuracy

In [44]:
def predict(data, expert_labels, stage_cols):
    acc = 0
    deepmapper_prediction = []
    for row in data:
        img = row[0]
        # model preds
        preds = [float(x) for x in row[1:]]

        # get expert label data
        match_ = expert_labels.loc[expert_labels["Image"] == img]
        #print(match_)
        if match_.empty:
            print("not found:", img)
            continue

        # get the expert label
        label = int(match_['label_idx'].iloc[0])
        #print("label",label)


        #get DeepMapper wound stage prediction
        pred = int(np.argmax(preds))
        #print(img, pred, label)

        #calculate accuracy
        if pred == label:
          acc += 1

        deepmapper_prediction.append(pred)
        #print(acc/len(data))

    return deepmapper_prediction, acc/len(data)

In [45]:
preds, acc = predict(data, expert_labels, stage_cols)
print(acc)
print(preds)

0.25
[0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


## Computer Averages for Adult

In [72]:
a_path = "./deepmapper_on_testing_mouse_imgage/adult"
a_csv_files = glob.glob(f"{a_path}/**/*.csv", recursive = True)

In [73]:
metrics = []

for f_name in a_csv_files:

    rows = []
    with open(f_name, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        data = list(reader)

    _, acc = predict(data, expert_labels, stage_cols)

    metrics.append(acc)

In [74]:
metrics = np.array(metrics)
float(np.mean(metrics)), float(np.std(metrics))


(0.365, 0.07219764539096826)

## Computer Averages for Young

In [80]:
y_path = "./deepmapper_on_testing_mouse_imgage/young"
y_csv_files = glob.glob(f"{y_path}/**/*.csv", recursive = True)

In [85]:
metrics = []

for f_name in y_csv_files:

    rows = []
    with open(f_name, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        data = list(reader)

    _, acc = predict(data, expert_labels, stage_cols)

    metrics.append(acc)

In [86]:
metrics = np.array(metrics)
float(np.mean(metrics)), float(np.std(metrics))

(0.665625, 0.04871392896287467)

### Metrics on all data:

In [87]:
data_path = "./deepmapper_on_testing_mouse_imgage/"
csv_files = glob.glob(f"{data_path}/**/*.csv", recursive = True)

In [90]:
metrics = []

for f_name in csv_files:

    rows = []
    with open(f_name, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        data = list(reader)

    _, acc = predict(data, expert_labels, stage_cols)

    metrics.append(acc)

In [91]:
metrics = np.array(metrics)
float(np.mean(metrics)), float(np.std(metrics))

(0.5153125, 0.16243959213735426)